# E2A-SML on Google Colab

Run the first cell to install E2A-SML and open its web GUI. Select a GPU runtime first if Colab offers one. The **small** BookNLP model is a good starting point on the free tier.

The GUI link is public while this runtime is active. [Free Colab runtimes may end web UI sessions](https://research.google.com/colaboratory/faq.html); the optional CLI cell below works in the notebook. Download your SML file before the runtime ends.


In [ ]:
#@title Install and open the E2A-SML web GUI
launch_web_gui = True #@param {type:"boolean"}

import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import HTML, display

repo = Path(os.environ.get("E2A_SML_DIR", "/content/E2A-SML"))
ref = os.environ.get("E2A_SML_REF", "main")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", ref,
                    "https://github.com/DrewThomasson/E2A-SML.git", str(repo)], check=True)

if not shutil.which("ebook-convert"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "calibre"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    raise RuntimeError("uv was installed but is not on PATH")
venv_python = repo / ".venv" / "bin" / "python"
if not venv_python.exists():
    subprocess.run([uv, "venv", "--python", "3.10", str(repo / ".venv")], check=True)
subprocess.run([uv, "pip", "install", "--python", str(venv_python),
                "-r", str(repo / "requirements.txt")], check=True)
model_url = subprocess.check_output(
    [str(venv_python), "-m", "spacy", "info", "en_core_web_sm", "--url"], text=True
).strip()
subprocess.run([uv, "pip", "install", "--python", str(venv_python), model_url], check=True)
subprocess.run([str(venv_python), "-c",
                "from sml_extractor.core import check_booknlp_installation; "
                "ok, message = check_booknlp_installation(); print(message); assert ok, message"],
               cwd=repo, check=True)
subprocess.run([str(venv_python), "-c",
                "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (CPU mode)')"],
               check=True)

if launch_web_gui:
    log_path = repo / "data" / "colab-gui.log"
    log_path.parent.mkdir(exist_ok=True)
    with log_path.open("w") as log:
        gui = subprocess.Popen(
            [str(venv_python), "-u", "cli.py", "--gui", "--host", "0.0.0.0", "--share"],
            cwd=repo, stdout=log, stderr=subprocess.STDOUT,
            env={**os.environ, "SML_DATA_DIR": str(repo / "data")},
        )
    for _ in range(180):
        output = log_path.read_text(errors="replace")
        match = re.search(r"https://[a-z0-9-]+\.gradio\.live", output)
        if match:
            url = match.group(0)
            print("Web GUI:", url)
            display(HTML(f'<a href="{url}" target="_blank">Open E2A-SML</a>'))
            break
        if gui.poll() is not None:
            raise RuntimeError("GUI stopped during startup. See " + str(log_path) + "\n" + output[-2000:])
        time.sleep(1)
    else:
        raise TimeoutError("No Gradio link appeared. See " + str(log_path))


## Optional CLI

Run this cell if you prefer a notebook upload and direct SML download.


In [ ]:
#@title Optional: process a book through the CLI and download its SML file
model_size = "small" #@param ["small", "big"]
from google.colab import files
from pathlib import Path
import os
import subprocess

repo = Path(os.environ.get("E2A_SML_DIR", "/content/E2A-SML"))
venv_python = repo / ".venv" / "bin" / "python"
if not venv_python.exists():
    raise RuntimeError("Run the install cell first (you can uncheck launch_web_gui).")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No book was uploaded")
book = Path.cwd() / next(iter(uploaded))
output_dir = repo / "output"
subprocess.run([str(venv_python), "cli.py", str(book), "--model", model_size,
                "--output-dir", str(output_dir)], cwd=repo,
               env={**os.environ, "SML_DATA_DIR": str(repo / "data")}, check=True)
files.download(str(output_dir / (book.stem + ".e2a.sml.txt")))
